# Namo Turn Detector — Fine-tune (o'zbekcha)

**LiveKit'dan farqi:** Namo — *sequence classification* modeli (2 sinf: 0=davom, 1=tugagan),
causal LM emas. Shuning uchun:
- `AutoModelForSequenceClassification` (num_labels=2), **cross-entropy loss** (standart, maxsus loss yo'q)
- LoRA `task_type="SEQ_CLS"`, `target_modules=["Wqkv","Wo","Wi"]` (ModernBERT nomlari), `modules_to_save=["classifier","head"]`

**2 muhim nozik joy:**
1. Namo repo'sida faqat ONNX bor (PyTorch vazn YO'Q) -> aynan Namo'dan davom ettirib bo'lmaydi.
   Bir xil base — **`jhu-clsp/mmBERT-base`** (PyTorch bor) — dan o'qitamiz. Arxitektura/tokenizer aynan bir xil.
2. Namo bitta matn oladi (kontekstsiz). Bizning dataset kontekstga bog'liq -> suhbatni bitta matnga
   jamlaymiz (`Agent:/User:`). Bizning label 1=tugagan Namo'ning 1=End of Turn bilan mos.


In [5]:
# BASE model: Namo bilan bir xil arxitektura (jhu-clsp/mmBERT-base).
BASE = "jhu-clsp/mmBERT-base"
MAX_LEN = 256
DATA_DIR = "../uz_dataset"      # notebook Namo_Turn/ ichida -> dataset bir pog'ona yuqorida
# SO'Z fayllaridan foydalanamiz; raqamga aylantirish + bo'shliq to_text ichida bo'ladi
# (shunda train va inference KAFOLATLANGAN bir xil -> spacing muammosi yo'q).
TRAIN = f"{DATA_DIR}/uz_eou_train.jsonl"
VAL   = f"{DATA_DIR}/uz_eou_val.jsonl"
SMOKE = f"{DATA_DIR}/uz_eou_smoke_test.jsonl"

import re, unicodedata, json
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(BASE)

# --- raqam normalizatsiyasi: so'z->raqam, keyin HAR RAQAMNI bo'shliq bilan ajratish ---
# Sabab: "0042055" va "00420 55" bir xil 7 raqam, lekin tokenizer ularni har xil
# bo'laklaydi -> model overfit bo'ladi. Har raqamni bo'shliq bilan ajratsak, istalgan
# spacing bir xil kanonik ko'rinishga keladi: "0 0 4 2 0 5 5".
ZERO={"nol","no'l","nul"}
UNITS={"nol":0,"no'l":0,"nul":0,"bir":1,"ikki":2,"uch":3,"to'rt":4,"besh":5,"olti":6,"yetti":7,"sakkiz":8,"to'qqiz":9}
TENS={"o'n":10,"yigirma":20,"o'ttiz":30,"qirq":40,"ellik":50,"oltmish":60,"yetmish":70,"sakson":80,"to'qson":90}
HUND={"yuz":100}; _LIK=("talik","taligi","ligi","lig'i","lig","lik")
def _norm(t):
    for c in "\u02bb\u02bc\u2018\u2019`\u00b4": t=t.replace(c,"'")
    t=unicodedata.normalize("NFKC",t.lower())
    t="".join(ch for ch in t if not(unicodedata.category(ch).startswith("P") and ch not in ["'","-"]))
    return re.sub(r"\s+"," ",t).strip()
def _lik(x):
    if x in UNITS or x in TENS or x in HUND: return x,False
    for s in _LIK:
        st=x[:-len(s)]
        if x.endswith(s) and (st in UNITS or st in TENS or st in HUND): return st,True
    return x,False
def convert_numbers(text):
    result,buf=[],[]; state={"cur":0,"st":False,"su":False,"act":False}
    def fg():
        if state["act"]: buf.append(str(state["cur"]))
        state.update(cur=0,st=False,su=False,act=False)
    def fr():
        fg()
        if buf: result.append("".join(buf)); buf.clear()
    for raw in _norm(text).split():
        if raw.isdigit(): fg(); buf.append(raw); continue
        x,ff=_lik(raw)
        if x in ZERO: fg(); buf.append("0")
        elif x in UNITS:
            if state["su"]: fg()
            state["cur"]+=UNITS[x]; state["su"]=state["act"]=True
        elif x in TENS:
            if state["st"] or state["su"]: fg()
            state["cur"]+=TENS[x]; state["st"]=state["act"]=True
        elif x in HUND:
            if not state["act"]: state["cur"]=1
            state["cur"]*=100; state["st"]=state["su"]=False; state["act"]=True
        else: fr(); result.append(raw); continue
        if ff: fg()
    fr()
    return " ".join(result)
def normalize_for_model(text):
    t = convert_numbers(text)
    t = re.sub(r"(?<=\d)(?=\d)", " ", t)     # har raqamni ajratamiz
    return re.sub(r"\s+"," ",t).strip()

def to_text(messages):
    """Suhbatni bitta matnga jamlaydi + raqam normalizatsiyasi (train=inference)."""
    parts=[]
    for m in messages:
        who = "Agent" if m["role"]=="assistant" else "User"
        parts.append(f"{who}: {normalize_for_model(m['content'])}")
    return "\n".join(parts)

ex=json.loads(open(TRAIN,encoding="utf-8").readline())
print("namuna matn:\n", to_text(ex["messages"]))
print("\nspacing testi (bir xil bo'lishi kerak):")
print(" ", normalize_for_model("passport seriyam ae 00420 55"))
print(" ", normalize_for_model("passport seriyam ae 0042055"))


namuna matn:
 Agent: bank axborot xizmati assalomu alaykum
User: kartamdan noma'lum to'lov yechilibdi
Agent: albatta shaxsingizni tasdiqlash uchun id karta seriya raqamingizni ayting
User: a be 2 4 4 1 2 9 0

spacing testi (bir xil bo'lishi kerak):
  passport seriyam ae 0 0 4 2 0 5 5
  passport seriyam ae 0 0 4 2 0 5 5


In [6]:
# (ixtiyoriy) Stock Namo ONNX bilan bazaviy ko'rsatkich — o'zbekchada qanchalik zaif.
# Diqqat: stock Namo kontekstsiz, o'zbekchaga o'qitilmagan -> past bo'lishi kutiladi.
import numpy as np, onnxruntime as ort
from huggingface_hub import hf_hub_download
NAMO = "videosdk-live/Namo-Turn-Detector-v1-Multilingual"
onnx_path = hf_hub_download(NAMO, "model_quant.onnx")
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
def namo_prob(text):
    enc = tok(text, truncation=True, max_length=MAX_LEN, return_tensors="np")
    logits = sess.run(None, {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]})[0][0]
    e = np.exp(logits - logits.max()); p = e / e.sum()
    return float(p[1])

rows = [json.loads(l) for l in open(SMOKE, encoding="utf-8")]
import numpy as np
probs = np.array([namo_prob(to_text(r["messages"])) for r in rows])
labels = np.array([r["label"] for r in rows])
from sklearn.metrics import roc_auc_score, accuracy_score
print("STOCK Namo (o'zbekcha smoke):")
print("  AUC:", round(roc_auc_score(labels, probs), 4))
print("  acc@0.5:", round(accuracy_score(labels, (probs>=0.5).astype(int)), 4))


STOCK Namo (o'zbekcha smoke):
  AUC: 0.429
  acc@0.5: 0.4771


In [7]:
import json, torch
from torch.utils.data import Dataset
from transformers import DataCollatorWithPadding

class TurnDataset(Dataset):
    def __init__(self, path):
        self.rows = [json.loads(l) for l in open(path, encoding="utf-8")]
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        enc = tok(to_text(r["messages"]), truncation=True, max_length=MAX_LEN)
        enc["labels"] = int(r["label"])          # SEQ_CLS -> butun son (0/1)
        return enc

collate = DataCollatorWithPadding(tok)            # standart right-padding
print("train:", len(TurnDataset(TRAIN)), " val:", len(TurnDataset(VAL)))


train: 11679  val: 1297


In [8]:
import torch
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model

model = AutoModelForSequenceClassification.from_pretrained(
    BASE, num_labels=2,
    id2label={0: "not_end_of_turn", 1: "end_of_turn"},
    label2id={"not_end_of_turn": 0, "end_of_turn": 1},
)
# classifier head yangi (mmBERT-base'da yo'q edi) — LoRA + head to'liq o'qitiladi
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["Wqkv", "Wo", "Wi"],   # ModernBERT: attn.Wqkv/Wo, mlp.Wi/Wo
    modules_to_save=["classifier", "head"],# yangi classifier + pooling head to'liq o'qitiladi
    task_type="SEQ_CLS",
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: jhu-clsp/mmBERT-base
Key               | Status     | 
------------------+------------+-
decoder.weight    | UNEXPECTED | 
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 3,971,330 || all params: 311,503,108 || trainable%: 1.2749


In [ ]:
import numpy as np
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

def compute_metrics(ep):
    logits, labels = ep
    preds = logits.argmax(-1)
    e = np.exp(logits - logits.max(-1, keepdims=True)); probs = (e / e.sum(-1, keepdims=True))[:, 1]
    return {"accuracy": accuracy_score(labels, preds),
            "f1": f1_score(labels, preds),
            "auc": roc_auc_score(labels, probs)}

args = TrainingArguments(
    output_dir="./namo_uz_lora",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    learning_rate=2e-4,                 
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
)
trainer = Trainer(
    model=model, args=args,
    train_dataset=TurnDataset(TRAIN),
    eval_dataset=TurnDataset(VAL),
    data_collator=collate,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer.train()


In [ ]:
trainer.save_model("./namo_uz_lora/final")
tok.save_pretrained("./namo_uz_lora/final")
print("saqlandi -> ./namo_uz_lora/final")


In [ ]:
import numpy as np, torch
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

@torch.no_grad()
def predict_prob(messages):
    enc = tok(to_text(messages), truncation=True, max_length=MAX_LEN, return_tensors="pt")
    enc = {k: v.to(model.device) for k, v in enc.items()}
    logits = model(**enc).logits[0]
    return torch.softmax(logits, -1)[1].item()    # P(tugagan / End of Turn)

model.eval()
for name, path in [("VAL", VAL), ("SMOKE", SMOKE)]:
    rows = [json.loads(l) for l in open(path, encoding="utf-8")]
    probs = np.array([predict_prob(r["messages"]) for r in rows])
    labels = np.array([r["label"] for r in rows])
    print(f"\n=== {name} ({len(rows)}) ===")
    print("AUC:", round(roc_auc_score(labels, probs), 4),
          " acc@0.5:", round(accuracy_score(labels, (probs>=0.5).astype(int)), 4),
          " f1@0.5:", round(f1_score(labels, (probs>=0.5).astype(int)), 4))

# smoke — pattern bo'yicha xatolar
rows = [json.loads(l) for l in open(SMOKE, encoding="utf-8")]
print("\nsmoke xatolar (@0.5):")
for r in rows:
    p = predict_prob(r["messages"])
    if (p>=0.5) != (r["label"]==1):
        print(f"  kut={r['label']} olindi={p:.2f} [{r.get('pattern')}] {r['messages'][-1]['content'][:55]}")


## Deploy va eslatmalar

- Bu model KONTEKST bilan ishlaydi (`to_text` — Agent:/User:). Inference'da ham xuddi shu
  formatni ber (stock Namo'dan farqli — u bitta utterance olardi).
- ONNX eksport: `optimum-cli export onnx --model ./namo_uz_lora/final ./namo_uz_lora/onnx --task text-classification`
  (avval `merge_and_unload()` bilan LoRA'ni base'ga qo'shib, keyin eksport).
- Full fine-tune xohlasang: LoRA kataklarini olib tashla, to'g'ridan-to'g'ri `model` ni Trainer'ga ber,
  `learning_rate=2e-5` qil (past LR).
- Raqamli (v2) o'rniga so'z (v1) sinamoqchi bo'lsang: TRAIN/VAL/SMOKE dagi `_num` ni olib tashla.
